<a href="https://colab.research.google.com/github/Ruc6/Tokenization/blob/main/SOCLSTMEnergy_pred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded=files.upload()


Saving Energy_consumption(data).csv to Energy_consumption(data).csv


In [ ]:
import pandas as pd
df=pd.DataFrame(pd.read_csv('Energy_consumption(data).csv'))
df.head()

,Timestamp,Temperature,Humidity,SquareFootage,Occupancy,HVACUsage,LightingUsage,RenewableEnergy,DayOfWeek,Holiday,EnergyConsumption
0,2022-01-01 00:00:00,25.139433,43.431581,1565.693999,5,On,Off,2.774699,Monday,No,75.364373
1,2022-01-01 01:00:00,27.731651,54.225919,1411.064918,1,On,On,21.831384,Saturday,No,83.401855
2,2022-01-01 02:00:00,28.704277,58.907658,1755.715009,2,Off,Off,6.764672,Sunday,No,78.270888
3,2022-01-01 03:00:00,20.080469,50.371637,1452.316318,1,Off,On,8.623447,Wednesday,No,56.519850
4,2022-01-01 04:00:00,23.097359,51.401421,1094.130359,9,On,Off,3.071969,Friday,No,70.811732


In [ ]:
df.describe()


,Temperature,Humidity,SquareFootage,Occupancy,RenewableEnergy,EnergyConsumption
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,24.982026,45.395412,1500.052488,4.581000,15.132813,77.055873
std,2.836850,8.518905,288.418873,2.865598,8.745917,8.144112
min,20.007565,30.015975,1000.512661,0.000000,0.006642,53.263278
25%,22.645070,38.297722,1247.108548,2.000000,7.628385,71.544690
50%,24.751637,45.972116,1507.967426,5.000000,15.072296,76.943696
75%,27.418174,52.420066,1740.340165,7.000000,22.884064,82.921742
max,29.998671,59.969085,1999.982252,9.000000,29.965327,99.201120


In [ ]:
df.shape

(1000, 11)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler=MinMaxScaler()
df=pd.DataFrame(scaler.fit_transform(df.select_dtypes(include=['number'])),columns=df.select_dtypes(include=['number']).columns)
df.head()

,Temperature,Humidity,SquareFootage,Occupancy,RenewableEnergy,EnergyConsumption
0,0.513644,0.447887,0.565481,0.555556,0.092396,0.481109
1,0.773096,0.808261,0.410770,0.111111,0.728495,0.656073
2,0.870445,0.964564,0.755603,0.222222,0.225578,0.544379
3,0.007297,0.679584,0.452043,0.111111,0.287623,0.070891
4,0.309254,0.713964,0.093667,1.000000,0.102318,0.382004


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('EnergyConsumption', axis=1)
y = df['EnergyConsumption']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(800, 5) (800,) (200, 5) (200,)


In [ ]:
class LSTM(nn.Module):
   def __init__(self,input_size=5,hidden_layer_size=10,output_size=1):
       super().__init__()
       self.W1=nn.Linear(input_size,hidden_layer_size)
       self.W2=nn.Linear(output_size,hidden_layer_size)
       self.b1=torch.zeros(hidden_layer_size)
       self.W3=nn.Linear(input_size,hidden_layer_size)
       self.W4=nn.Linear(output_size,hidden_layer_size)
       self.b2=torch.zeros(hidden_layer_size)
       self.W5=nn.Linear(input_size,hidden_layer_size)
       self.W6=nn.Linear(output_size,hidden_layer_size)
       self.b3=torch.zeros(hidden_layer_size)
       self.W7=nn.Linear(input_size,hidden_layer_size)
       self.W8=nn.Linear(output_size,hidden_layer_size)
       self.b4=torch.zeros(hidden_layer_size)
    def forward(self,x):
        shortM=torch.zeros(output_size)
        longM=torch.zeros(output_size)
                seq_len=x.shape[0]
                for t in range(seq_len):
                    shortM.unsqueeze(0)
                    xt=x[t]
                    h1=xt@self.W1+shortM@self.W2+self.b1
                    h1=torch.sigmoid(h1)
                    longM=longM*h1
                    h2=xt@self.W3+ShtorM@self.W4+self.b2
                    h2=torch.sigmoid(h2)
                    h3=xt@self.W5+shortM@self.W6+self.b3
                    h3=torch.tanh(h3)
                    h4=h2*h3
                    longM+=h4
                    h5=xt@self.W7+shortM@self.W8+self.b4
                    h5=torch.sigmoid(h5)
                    h6=torch.tanh(longM)
                    h5=h5*h6
                    shortM=h5
                return shortM
model = LSTM()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
loss_fn = nn.MSELoss()
for epoch in range(100):
    y_pred=model(X_train)
    loss=loss_fn(y_pred,y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f'Epoch: {epoch}, Loss: {loss.item()}')

In [ ]:
@torch.no_grad()
y_pred=model(X_test)
loss=loss_fn(y_pred,y_test)
print(f'Loss: {loss.item()}')

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# -------------------------------
# Read Data
# -------------------------------

df = pd.read_csv("Energy_consumption(data).csv")

# -------------------------------
# Min-Max Scaling
# -------------------------------

scaler = MinMaxScaler()

df = pd.DataFrame(
    scaler.fit_transform(df.select_dtypes(include=['number'])),
    columns=df.select_dtypes(include=['number']).columns
)

# -------------------------------
# Split Features & Target
# -------------------------------

X = df.drop("EnergyConsumption", axis=1)
y = df["EnergyConsumption"]

# IMPORTANT: keep sequence order

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

# -------------------------------
# Convert to Tensor
# -------------------------------

X_train = torch.tensor(X_train.values, dtype=torch.float32)
X_test = torch.tensor(X_test.values, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.float32)


In [ ]:

class LSTM(nn.Module):

    def __init__(self, input_size=5, hidden_layer_size=10):

        super().__init__()

        self.hidden_layer_size = hidden_layer_size

        # Forget Gate
        self.W1 = nn.Linear(input_size, hidden_layer_size, bias=False)
        self.W2 = nn.Linear(hidden_layer_size, hidden_layer_size, bias=False)
        self.b1 = nn.Parameter(torch.zeros(hidden_layer_size))

        # Input Gate
        self.W3 = nn.Linear(input_size, hidden_layer_size, bias=False)
        self.W4 = nn.Linear(hidden_layer_size, hidden_layer_size, bias=False)
        self.b2 = nn.Parameter(torch.zeros(hidden_layer_size))

        # Candidate Memory
        self.W5 = nn.Linear(input_size, hidden_layer_size, bias=False)
        self.W6 = nn.Linear(hidden_layer_size, hidden_layer_size, bias=False)
        self.b3 = nn.Parameter(torch.zeros(hidden_layer_size))

        # Output Gate
        self.W7 = nn.Linear(input_size, hidden_layer_size, bias=False)
        self.W8 = nn.Linear(hidden_layer_size, hidden_layer_size, bias=False)
        self.b4 = nn.Parameter(torch.zeros(hidden_layer_size))

        # Prediction Layer
        self.fc = nn.Linear(hidden_layer_size, 1)

    def forward(self, x):

        shortM = torch.zeros(1, self.hidden_layer_size, device=x.device)
        longM = torch.zeros(1, self.hidden_layer_size, device=x.device)

        outputs = []

        seq_len = x.shape[0]

        for t in range(seq_len):

            xt = x[t].unsqueeze(0)

            # ---------------- Forget Gate ----------------

            h1 = self.W1(xt)
            h1 = h1 + self.W2(shortM)
            h1 = h1 + self.b1
            h1 = torch.sigmoid(h1)

            longM = longM * h1

            # ---------------- Input Gate ----------------

            h2 = self.W3(xt)
            h2 = h2 + self.W4(shortM)
            h2 = h2 + self.b2
            h2 = torch.sigmoid(h2)

            # ---------------- Candidate ----------------

            h3 = self.W5(xt)
            h3 = h3 + self.W6(shortM)
            h3 = h3 + self.b3
            h3 = torch.tanh(h3)

            h4 = h2 * h3

            longM = longM + h4

            # ---------------- Output Gate ----------------

            h5 = self.W7(xt)
            h5 = h5 + self.W8(shortM)
            h5 = h5 + self.b4
            h5 = torch.sigmoid(h5)

            h6 = torch.tanh(longM)

            shortM = h5 * h6

            # Prediction from hidden state
            prediction = self.fc(shortM)

            outputs.append(prediction.squeeze())

        outputs = torch.stack(outputs)

        return outputs


model = LSTM()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

loss_fn = nn.MSELoss()


epochs = 35

for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()

    y_pred = model(X_train)

    loss = loss_fn(y_pred, y_train)

    loss.backward()

    optimizer.step()

    print(f"Epoch {epoch+1:3d} | Loss = {loss.item():.6f}")


model.eval()

with torch.no_grad():

    y_pred = model(X_test)

    loss = loss_fn(y_pred, y_test)

    print("\nTest Loss :", loss.item())

Epoch   1 | Loss = 0.386281
Epoch   2 | Loss = 0.379854
Epoch   3 | Loss = 0.373492
Epoch   4 | Loss = 0.367187
Epoch   5 | Loss = 0.360933
Epoch   6 | Loss = 0.354725
Epoch   7 | Loss = 0.348560
Epoch   8 | Loss = 0.342436
Epoch   9 | Loss = 0.336350
Epoch  10 | Loss = 0.330301
Epoch  11 | Loss = 0.324287
Epoch  12 | Loss = 0.318305
Epoch  13 | Loss = 0.312354
Epoch  14 | Loss = 0.306431
Epoch  15 | Loss = 0.300534
Epoch  16 | Loss = 0.294660
Epoch  17 | Loss = 0.288807
Epoch  18 | Loss = 0.282973
Epoch  19 | Loss = 0.277153
Epoch  20 | Loss = 0.271347
Epoch  21 | Loss = 0.265550
Epoch  22 | Loss = 0.259761
Epoch  23 | Loss = 0.253977
Epoch  24 | Loss = 0.248196
Epoch  25 | Loss = 0.242415
Epoch  26 | Loss = 0.236631
Epoch  27 | Loss = 0.230843
Epoch  28 | Loss = 0.225050
Epoch  29 | Loss = 0.219248
Epoch  30 | Loss = 0.213437
Epoch  31 | Loss = 0.207615
Epoch  32 | Loss = 0.201781
Epoch  33 | Loss = 0.195934
Epoch  34 | Loss = 0.190075
Epoch  35 | Loss = 0.184202

Test Loss : 0.19075